In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# ── CONFIGURE HERE ─────────────────────────────────────────────────────────────
DATA_DIR = Path(r"C:\\Users\\Siddhartha\\Sem8\\MajorProject\\Experiments\\CyberSec")
OUT_DIR  = DATA_DIR / "_cleaned"
OUT_DIR.mkdir(parents=True, exist_ok=True)
# ────────────────────────────────────────────────────────────────────────────────

## CICIDS2017 — Cleaning Notebook

The **CIC-IDS-2017** dataset (Sharafaldin et al.) contains network flow records captured over
five working days (Monday 3 Jul → Friday 7 Jul 2017) at a simulated corporate network.

Each row is a **flow** described by 78 CICFlowMeter features plus a `Label` column.

**Important limitation of this release:**  
The publicly distributed CICFlowMeter CSV export **omits source/destination IP addresses**.
This means the "unique source IPs" KPI from the Phase 3 plan cannot be computed directly.
We substitute it with **`TOTAL_FLOWS`** (total flow count per day), which serves as a proxy
for network activity volume and has the same sensitivity structure (Δ = 1 per flow).

The three KPIs tracked are:
| KPI | Column derivation | Sensitivity Δ |
|---|---|---|
| `ALERT_COUNT` | non-BENIGN flows per day | 1 per flow |
| `TOTAL_FLOWS` | all flows per day | 1 per flow |
| `TOTAL_BYTES` | sum of (fwd + bwd packet bytes), clipped at 90th pct | B_CLIP per flow |

## Step 1 — Date Inference from Filenames

The CSVs carry no timestamp column; dates are inferred from the day-of-week prefix in each filename.
Multiple files can share the same day (Thursday and Friday each have morning + afternoon captures).
These are concatenated before aggregation.

In [2]:
# Canonical date mapping for CICIDS2017 capture week
# Reference: Sharafaldin et al., "Toward Generating a New Intrusion Detection Dataset ...", 2018
DATE_MAP = {
    "Monday":    pd.Timestamp("2017-07-03"),
    "Tuesday":   pd.Timestamp("2017-07-04"),
    "Wednesday": pd.Timestamp("2017-07-05"),
    "Thursday":  pd.Timestamp("2017-07-06"),
    "Friday":    pd.Timestamp("2017-07-07"),
}

def infer_date(filename: str) -> pd.Timestamp:
    """Return the capture date for a CICIDS2017 CSV based on its filename."""
    fname = Path(filename).name
    for day_key, ts in DATE_MAP.items():
        if fname.startswith(day_key):
            return ts
    raise ValueError(f"Cannot infer date from filename: {fname}")

# Discover all CICIDS2017 CSVs
csv_files = sorted(DATA_DIR.glob("*.pcap_ISCX.csv"))
assert len(csv_files) > 0, f"No *.pcap_ISCX.csv found in {DATA_DIR}"

print(f"Found {len(csv_files)} CSV files:")
for p in csv_files:
    print(f"  {p.name:65s}  date={infer_date(p).date()}")

Found 8 CSV files:
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv                   date=2017-07-07
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv               date=2017-07-07
  Friday-WorkingHours-Morning.pcap_ISCX.csv                          date=2017-07-07
  Monday-WorkingHours.pcap_ISCX.csv                                  date=2017-07-03
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv        date=2017-07-06
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv             date=2017-07-06
  Tuesday-WorkingHours.pcap_ISCX.csv                                 date=2017-07-04
  Wednesday-workingHours.pcap_ISCX.csv                               date=2017-07-05


In [3]:
# Load and concatenate all CSVs, tagging each row with its capture date.
# Column names in this dataset have inconsistent leading/trailing spaces —
# we strip them immediately on load.

dfs = []
for p in csv_files:
    df = pd.read_csv(p, low_memory=False)
    # Normalise column names: strip whitespace, replace inner spaces with underscores
    df.columns = [c.strip().replace(" ", "_").replace("/", "_per_").lower() for c in df.columns]
    df["date"] = infer_date(p)
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)
print(f"Combined shape: {raw.shape}")
print(f"Columns ({len(raw.columns)}):")
print(list(raw.columns))

Combined shape: (2830743, 80)
Columns (80):
['destination_port', 'flow_duration', 'total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytes_per_s', 'flow_packets_per_s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packets_per_s', 'bwd_packets_per_s', 'min_packet_length', 'max_packet_length', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', 'fin_flag_count', 'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag

## Step 2 — Column Selection and Feature Engineering

We keep only the columns needed for the three KPIs:
- `label` → derives `is_attack` (0 = BENIGN, 1 = any attack)
- `total_length_of_fwd_packets` + `total_length_of_bwd_packets` → `flow_bytes`
- `date` → carry-over from loading

In [4]:
# Identify the label column (normalised name)
LABEL_COL = "label"
assert LABEL_COL in raw.columns, f"Label column not found; available: {list(raw.columns)}"

# Print all unique labels across the week
print("Unique labels:")
for lbl, cnt in raw[LABEL_COL].value_counts().items():
    marker = "BENIGN" if lbl == "BENIGN" else "ATTACK"
    print(f"  [{marker}]  {lbl:45s}  {cnt:>10,}")

Unique labels:
  [BENIGN]  BENIGN                                          2,273,097
  [ATTACK]  DoS Hulk                                          231,073
  [ATTACK]  PortScan                                          158,930
  [ATTACK]  DDoS                                              128,027
  [ATTACK]  DoS GoldenEye                                      10,293
  [ATTACK]  FTP-Patator                                         7,938
  [ATTACK]  SSH-Patator                                         5,897
  [ATTACK]  DoS slowloris                                       5,796
  [ATTACK]  DoS Slowhttptest                                    5,499
  [ATTACK]  Bot                                                 1,966
  [ATTACK]  Web Attack � Brute Force                            1,507
  [ATTACK]  Web Attack � XSS                                      652
  [ATTACK]  Infiltration                                           36
  [ATTACK]  Web Attack � Sql Injection                             21
  [AT

In [5]:
# Derive the two byte columns (check for their normalised names)
FWD_BYTES_COL = "total_length_of_fwd_packets"
BWD_BYTES_COL = "total_length_of_bwd_packets"

for c in [FWD_BYTES_COL, BWD_BYTES_COL]:
    assert c in raw.columns, f"Expected column '{c}' not found; available: {list(raw.columns)}"

# Build the working dataframe with only the columns we need
flows = raw[["date", LABEL_COL, FWD_BYTES_COL, BWD_BYTES_COL]].copy()

# is_attack: 1 for any non-BENIGN label, 0 for BENIGN
flows["is_attack"] = (flows[LABEL_COL] != "BENIGN").astype("int8")

# flow_bytes: total bytes transferred in this flow
flows["flow_bytes"] = (
    pd.to_numeric(flows[FWD_BYTES_COL], errors="coerce").fillna(0).clip(lower=0) +
    pd.to_numeric(flows[BWD_BYTES_COL], errors="coerce").fillna(0).clip(lower=0)
)

flows = flows[["date", "is_attack", "flow_bytes"]].copy()
print(f"Working dataframe shape: {flows.shape}")
print(flows.dtypes)
print(flows.head(4))

Working dataframe shape: (2830743, 3)
date          datetime64[s]
is_attack              int8
flow_bytes            int64
dtype: object
        date  is_attack  flow_bytes
0 2017-07-07          0          12
1 2017-07-07          0          12
2 2017-07-07          0          12
3 2017-07-07          0          12


## Step 3 — Data Quality Checks

CICIDS2017 is known to contain a small number of **infinite values** in rate-based columns
(e.g. `flow_bytes_per_s` when flow duration = 0). We already replaced those with numeric
columns that don't suffer this issue, but we verify anyway.

In [6]:
# Check for NaNs and infinite values in our working columns
n_null  = flows["flow_bytes"].isna().sum()
n_inf   = np.isinf(flows["flow_bytes"].replace([np.inf, -np.inf], np.nan).fillna(0)).sum()
n_neg   = (flows["flow_bytes"] < 0).sum()

print(f"NaN flow_bytes: {n_null:,}")
print(f"Inf flow_bytes: {n_inf:,}")
print(f"Negative flow_bytes: {n_neg:,}")

# Replace any remaining NaNs or infinities with 0
flows["flow_bytes"] = flows["flow_bytes"].replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)

print("\nflow_bytes distribution (bytes):")
print(flows["flow_bytes"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]).round(2))

NaN flow_bytes: 0
Inf flow_bytes: 0
Negative flow_bytes: 0

flow_bytes distribution (bytes):
count    2.830743e+06
mean     1.671194e+04
std      2.266643e+06
min      0.000000e+00
50%      2.100000e+02
75%      1.261000e+03
90%      1.166300e+04
95%      1.199000e+04
99%      7.690958e+04
max      6.567764e+08
Name: flow_bytes, dtype: float64


## Step 4 — Byte Sensitivity Calibration

We compute **B_CLIP**: the 90th percentile of per-flow byte counts.  
This is the sensitivity Δ used for the `TOTAL_BYTES` KPI in all DP experiments.
Flows with bytes > B_CLIP are clipped before aggregation — this bounds the worst-case
influence of any single flow on the daily byte sum.

In [7]:
B_CLIP = float(flows["flow_bytes"].quantile(0.90))
print(f"B_CLIP (90th pct of per-flow bytes) = {B_CLIP:,.2f} bytes")

# Fraction of flows affected by clipping
frac_clipped = (flows["flow_bytes"] > B_CLIP).mean()
print(f"Fraction of flows that will be clipped: {frac_clipped:.2%}")

# Plot byte distribution
fig, ax = plt.subplots(figsize=(8, 3))
clipped = flows["flow_bytes"].clip(upper=B_CLIP * 3)
ax.hist(clipped, bins=80, color="#4e79a7", edgecolor="none", alpha=0.8)
ax.axvline(B_CLIP, color="#e15759", lw=2, ls="--", label=f"B_CLIP = {B_CLIP:,.0f}")
ax.set_xlabel("Per-flow bytes (clipped at 3×B_CLIP for display)")
ax.set_ylabel("Flow count")
ax.set_title("CICIDS2017 — Per-flow byte distribution")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "byte_distribution.png", dpi=120)
plt.close()
print("Saved byte_distribution.png")

B_CLIP (90th pct of per-flow bytes) = 11,663.00 bytes
Fraction of flows that will be clipped: 9.97%
Saved byte_distribution.png


## Step 5 — Daily Aggregation

Aggregate flow-level records to **daily KPIs**.  
Per-flow bytes are clipped at B_CLIP before summing so that the daily sensitivity is bounded.

In [8]:
# Clip flow_bytes at B_CLIP before summing → bounded sensitivity
flows["flow_bytes_clipped"] = flows["flow_bytes"].clip(upper=B_CLIP)

daily = (
    flows.groupby("date")
    .agg(
        ALERT_COUNT=("is_attack",          "sum"),    # non-BENIGN flows
        TOTAL_FLOWS=("is_attack",          "count"),  # all flows (proxy for unique src IPs)
        TOTAL_BYTES=("flow_bytes_clipped", "sum"),    # total bounded bytes
    )
    .sort_index()
)

# Reindex to the full 5-day capture window (fill any missing days with 0)
full_idx = pd.date_range("2017-07-03", "2017-07-07", freq="D")
daily = daily.reindex(full_idx).fillna(0).astype({"ALERT_COUNT": int, "TOTAL_FLOWS": int})
daily.index.name = "date"

print("Daily KPIs:")
display(daily)
print(f"\nAlert rate per day: {(daily['ALERT_COUNT']/daily['TOTAL_FLOWS']*100).round(2).to_dict()}")

Daily KPIs:


,ALERT_COUNT,TOTAL_FLOWS,TOTAL_BYTES
date,,,
2017-07-03,0,529918,784989316
2017-07-04,13835,445909,676831226
2017-07-05,252672,692703,2571366171
2017-07-06,2216,458968,589786931
2017-07-07,288923,703245,1680125796



Alert rate per day: {Timestamp('2017-07-03 00:00:00'): 0.0, Timestamp('2017-07-04 00:00:00'): 3.1, Timestamp('2017-07-05 00:00:00'): 36.48, Timestamp('2017-07-06 00:00:00'): 0.48, Timestamp('2017-07-07 00:00:00'): 41.08}


## Step 6 — Save Outputs

Two files are written to `_cleaned/`:

| File | Description |
|---|---|
| `cicids_flows_canonical.parquet` | Flow-level: `date`, `is_attack`, `flow_bytes` |
| `cicids_daily.parquet` | Daily aggregates: `ALERT_COUNT`, `TOTAL_FLOWS`, `TOTAL_BYTES` |
| `cicids_sensitivity.json` | B_CLIP and other sensitivity metadata |

In [9]:
import json as _json

flows_out = OUT_DIR / "cicids_flows_canonical.parquet"
daily_out = OUT_DIR / "cicids_daily.parquet"
meta_out  = OUT_DIR / "cicids_sensitivity.json"

flows[["date", "is_attack", "flow_bytes"]].to_parquet(flows_out, index=False)
daily.to_parquet(daily_out)

meta = {
    "B_CLIP": B_CLIP,
    "bytes_clip_pct": 90.0,
    "capture_dates": [str(d.date()) for d in full_idx],
    "total_flows": int(flows.shape[0]),
    "total_attack_flows": int(flows["is_attack"].sum()),
    "sensitivities": {
        "ALERT_COUNT": 1,
        "TOTAL_FLOWS": 1,
        "TOTAL_BYTES": B_CLIP,
    },
    "kpi_note": (
        "TOTAL_FLOWS is used as a proxy for unique source IPs, "
        "which are not available in the CICFlowMeter CSV export."
    ),
}
with open(meta_out, "w") as f:
    _json.dump(meta, f, indent=2)

print("Saved:")
print(f"  {flows_out}")
print(f"  {daily_out}")
print(f"  {meta_out}")

Saved:
  C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\cicids_flows_canonical.parquet
  C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\cicids_daily.parquet
  C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\cicids_sensitivity.json


In [10]:
# ── Sanity checks ────────────────────────────────────────────────────────────
daily_check = pd.read_parquet(daily_out)
assert set(daily_check.columns) == {"ALERT_COUNT", "TOTAL_FLOWS", "TOTAL_BYTES"}, \
    f"Unexpected columns: {daily_check.columns}"
assert len(daily_check) == 5, f"Expected 5 days, got {len(daily_check)}"
assert (daily_check >= 0).all().all(), "Negative values found"
print("All sanity checks passed ✓")

# ── Daily KPI plot ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
kpi_colors = {"ALERT_COUNT": "#e15759", "TOTAL_FLOWS": "#4e79a7", "TOTAL_BYTES": "#59a14f"}

for ax, kpi in zip(axes, ["ALERT_COUNT", "TOTAL_FLOWS", "TOTAL_BYTES"]):
    ax.bar(daily_check.index, daily_check[kpi], color=kpi_colors[kpi], alpha=0.85)
    ax.set_ylabel(kpi.replace("_", " ").title(), fontsize=9)
    ax.tick_params(axis="x", rotation=30)

axes[0].set_title("CICIDS2017 — Daily KPI Ground Truth (bounded)")
plt.tight_layout()
plt.savefig(OUT_DIR / "daily_kpis.png", dpi=120)
plt.close()
print("Saved daily_kpis.png")
display(daily_check)

All sanity checks passed ✓
Saved daily_kpis.png


,ALERT_COUNT,TOTAL_FLOWS,TOTAL_BYTES
date,,,
2017-07-03,0,529918,784989316
2017-07-04,13835,445909,676831226
2017-07-05,252672,692703,2571366171
2017-07-06,2216,458968,589786931
2017-07-07,288923,703245,1680125796
